# 🧠 File 03 — Xây dựng mô hình (Model Building)
**Brain Tumor Classification MRI Dataset**

File này sẽ xây dựng **2 mô hình** để so sánh:
- **Model 1:** CNN tự xây (Baseline) — đơn giản, hiểu rõ từng bước
- **Model 2:** Transfer Learning với EfficientNetB0 — mạnh hơn, chính xác hơn

Sau đó so sánh kết quả để chọn model tốt nhất.

## 📦 Phần A — Import thư viện

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import (
    ModelCheckpoint,    # Lưu model tốt nhất
    EarlyStopping,      # Dừng sớm khi không cải thiện
    ReduceLROnPlateau   # Giảm learning rate khi bị tắc
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from pathlib import Path

# Kiểm tra GPU
gpus = tf.config.list_physical_devices('GPU')
print(f'✅ TensorFlow: {tf.__version__}')
print(f'   GPU available: {len(gpus) > 0}')
if gpus:
    print(f'   GPU: {gpus[0].name}')
else:
    print('   Đang dùng CPU — train sẽ chậm hơn, hãy kiên nhẫn!')

## ⚙️ Phần B — Load lại cấu hình và dữ liệu từ file 02

In [ ]:
# ============================================================
# CONFIG - must match 02_Preprocessing_v2.ipynb
# ============================================================
PROJECT_ROOT = Path.cwd()
DATA_DIR  = PROJECT_ROOT / 'data'
TRAIN_DIR = DATA_DIR / 'Training'
TEST_DIR  = DATA_DIR / 'Testing'
CSV_DIR   = PROJECT_ROOT

IMG_SIZE    = (240, 240)
BATCH_SIZE  = 32
VAL_SPLIT   = 0.2
RANDOM_SEED = 42
NUM_CLASSES = 4
EPOCHS_CNN  = 30
EPOCHS_TL   = 25

CLASSES = ['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']
CLASS_LABELS = {
    'glioma_tumor':     'Glioma',
    'meningioma_tumor': 'Meningioma',
    'no_tumor':         'Khong co u',
    'pituitary_tumor':  'Pituitary'
}

assert TRAIN_DIR.exists(), f'Missing training folder: {TRAIN_DIR}'
assert TEST_DIR.exists(),  f'Missing testing folder: {TEST_DIR}'


def build_dataframe(base_dir, classes):
    records = []
    for cls in classes:
        cls_path = base_dir / cls
        for ext in ('*.jpg', '*.jpeg', '*.png'):
            for img_path in cls_path.glob(ext):
                records.append({'filepath': str(img_path.resolve()), 'label': cls})
    df = pd.DataFrame(records)
    if df.empty:
        raise ValueError(f'No images found under {base_dir}')
    return df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)


def csvs_are_valid(*dfs):
    for df in dfs:
        if not {'filepath', 'label'}.issubset(df.columns):
            return False
        if set(df['label'].unique()) - set(CLASSES):
            return False
        if not df['filepath'].map(lambda p: Path(p).exists()).all():
            return False
    return True

csv_paths = [CSV_DIR / 'df_train.csv', CSV_DIR / 'df_val.csv', CSV_DIR / 'df_test.csv']
if all(p.exists() for p in csv_paths):
    df_train = pd.read_csv(csv_paths[0])
    df_val   = pd.read_csv(csv_paths[1])
    df_test  = pd.read_csv(csv_paths[2])
else:
    df_train = df_val = df_test = None

if df_train is None or not csvs_are_valid(df_train, df_val, df_test):
    print('CSV files missing/invalid. Rebuilding split from ./data ...')
    df_train_all = build_dataframe(TRAIN_DIR, CLASSES)
    df_test      = build_dataframe(TEST_DIR, CLASSES)
    df_train, df_val = train_test_split(
        df_train_all,
        test_size=VAL_SPLIT,
        random_state=RANDOM_SEED,
        stratify=df_train_all['label']
    )
    df_train = df_train.reset_index(drop=True)
    df_val   = df_val.reset_index(drop=True)
    df_train.to_csv(csv_paths[0], index=False)
    df_val.to_csv(csv_paths[1], index=False)
    df_test.to_csv(csv_paths[2], index=False)

print('Loaded data split:')
print(f'   Training   : {len(df_train)} images')
print(f'   Validation : {len(df_val)} images')
print(f'   Testing    : {len(df_test)} images')
print('Class distribution train:', df_train['label'].value_counts().reindex(CLASSES).to_dict())
print('Class distribution val  :', df_val['label'].value_counts().reindex(CLASSES).to_dict())
print('Class distribution test :', df_test['label'].value_counts().reindex(CLASSES).to_dict())

In [ ]:
# ============================================================
# Data generators
# Conservative augmentation is intentional: MRI tumor shape/location
# can be clinically meaningful, so do not over-randomize the image.
# ============================================================
from sklearn.utils import compute_class_weight

train_datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    shear_range=0.05,
    zoom_range=0.10,
    horizontal_flip=True,
    vertical_flip=False,
    brightness_range=[0.9, 1.1],
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator()

train_generator = train_datagen.flow_from_dataframe(
    dataframe=df_train, x_col='filepath', y_col='label',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=CLASSES,
    shuffle=True, seed=RANDOM_SEED
)
val_generator = val_test_datagen.flow_from_dataframe(
    dataframe=df_val, x_col='filepath', y_col='label',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=CLASSES,
    shuffle=False
)
test_generator = val_test_datagen.flow_from_dataframe(
    dataframe=df_test, x_col='filepath', y_col='label',
    target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=CLASSES,
    shuffle=False
)

CLASS_INDICES  = train_generator.class_indices
INDEX_TO_CLASS = {v: k for k, v in CLASS_INDICES.items()}
print(f'Class mapping: {CLASS_INDICES}')
print(f'Samples: train={train_generator.samples}, val={val_generator.samples}, test={test_generator.samples}')

labels = train_generator.classes
class_weights = compute_class_weight('balanced', classes=np.unique(labels), y=labels)
class_weight_dict = dict(enumerate(class_weights))

print('\nClass weights:')
for idx, cls in enumerate(CLASSES):
    print(f'   {CLASS_LABELS[cls]:<15}: {class_weight_dict[idx]:.3f}')

## 🏗️ Phần C — Model 1: CNN tự xây (Baseline)

Xây một mạng CNN từ đầu để hiểu cách hoạt động:
- **Conv2D + MaxPooling**: trích xuất đặc trưng (cạnh, texture, hình dạng)
- **BatchNormalization**: ổn định quá trình train
- **Dropout**: giảm overfitting
- **Dense**: phân loại cuối cùng

In [ ]:
# ============================================================
# Xây CNN Baseline
# ============================================================
def build_cnn_baseline(input_shape, num_classes):
    """
    CNN tự xây với 4 khối Conv.
    input_shape: (224, 224, 3)
    num_classes: 4
    """
    model = models.Sequential([
        layers.Rescaling(1./255, input_shape=input_shape),

        # ── Khối 1: Học đặc trưng cơ bản (cạnh, góc) ──
        layers.Conv2D(32, (3,3), activation='relu',
                      padding='same'),
        layers.BatchNormalization(),   # Chuẩn hoá để train ổn định hơn
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2, 2),     # Giảm kích thước ảnh xuống 1/2
        layers.Dropout(0.25),          # Bỏ 25% neuron ngẫu nhiên → giảm overfitting

        # ── Khối 2: Học đặc trưng phức tạp hơn ──
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # ── Khối 3: Học đặc trưng trừu tượng hơn ──
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.25),

        # ── Khối 4: Học đặc trưng cao cấp ──
        layers.Conv2D(256, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),
        layers.Dropout(0.3),

        # ── Phân loại ──
        layers.GlobalAveragePooling2D(), # Thay Flatten — giảm tham số, tránh overfit
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),             # Dropout mạnh hơn ở lớp fully connected
        layers.Dense(num_classes, activation='softmax')  # Softmax cho 4 class
    ], name='CNN_Baseline')

    return model

cnn_model = build_cnn_baseline((IMG_SIZE[0], IMG_SIZE[1], 3), NUM_CLASSES)

# Compile: chọn optimizer, loss function, metric
cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',  # Loss cho multi-class classification
    metrics=['accuracy']
)

cnn_model.summary()

In [ ]:
# ============================================================
# Định nghĩa Callbacks cho CNN Baseline
# ============================================================
callbacks_cnn = [
    # Lưu lại model có val_accuracy cao nhất
    ModelCheckpoint(
        'best_cnn_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    # Dừng train sớm nếu val_accuracy không tăng sau 8 epoch
    EarlyStopping(
        monitor='val_accuracy',
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    # Giảm learning rate nếu val_loss không giảm sau 4 epoch
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,       # Giảm xuống còn 50%
        patience=4,
        min_lr=1e-7,
        verbose=1
    )
]

print('✅ Callbacks đã sẵn sàng:')
print('   - ModelCheckpoint → lưu best_cnn_model.h5')
print('   - EarlyStopping   → dừng sau 8 epoch không cải thiện')
print('   - ReduceLROnPlateau → giảm LR sau 4 epoch tắc')

In [ ]:
# ============================================================
# Build + Train CNN Baseline
# ============================================================
cnn_model = build_cnn_baseline((IMG_SIZE[0], IMG_SIZE[1], 3), NUM_CLASSES)
cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
cnn_model.summary()

print('\n🚀 Bắt đầu train CNN Baseline...')
history_cnn = cnn_model.fit(
    train_generator,
    epochs=EPOCHS_CNN,
    validation_data=val_generator,
    callbacks=callbacks_cnn,
    class_weight=class_weight_dict,
    verbose=1
)
print('\n✅ Train CNN Baseline hoàn tất!')


## 📈 Phần D — Vẽ đồ thị quá trình train CNN

Xem accuracy và loss qua từng epoch để phát hiện:
- **Underfitting**: cả train và val đều thấp
- **Overfitting**: train cao nhưng val thấp/giảm

In [ ]:
# ============================================================
# Hàm vẽ đồ thị training history — dùng lại cho cả 2 model
# ============================================================
def plot_training_history(history, model_name, save_path=None):
    """
    Vẽ đồ thị Accuracy và Loss theo epoch.
    history   : kết quả trả về từ model.fit()
    model_name: tên model để đặt tiêu đề
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Quá trình Training — {model_name}',
                 fontsize=14, fontweight='bold')

    epochs_ran = range(1, len(history.history['accuracy']) + 1)

    # --- Accuracy ---
    axes[0].plot(epochs_ran, history.history['accuracy'],
                 'b-o', markersize=4, label='Train Accuracy')
    axes[0].plot(epochs_ran, history.history['val_accuracy'],
                 'r-o', markersize=4, label='Val Accuracy')
    axes[0].set_title('Accuracy theo Epoch')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].set_ylim([0, 1])
    # Đánh dấu epoch tốt nhất
    best_epoch = np.argmax(history.history['val_accuracy'])
    axes[0].axvline(x=best_epoch+1, color='green', linestyle='--',
                    alpha=0.7, label=f'Best epoch: {best_epoch+1}')
    axes[0].legend()

    # --- Loss ---
    axes[1].plot(epochs_ran, history.history['loss'],
                 'b-o', markersize=4, label='Train Loss')
    axes[1].plot(epochs_ran, history.history['val_loss'],
                 'r-o', markersize=4, label='Val Loss')
    axes[1].set_title('Loss theo Epoch')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'💾 Đã lưu: {save_path}')
    plt.show()

    # In kết quả tốt nhất
    best_val_acc = max(history.history['val_accuracy'])
    print(f'\n🏆 Val Accuracy tốt nhất: {best_val_acc:.4f} ({best_val_acc*100:.2f}%)')


plot_training_history(history_cnn, 'CNN Baseline', 'model_cnn_history.png')

## 🚀 Phần E — Model 2: Transfer Learning (EfficientNetB0)

**Transfer Learning** là dùng lại một mạng đã được train trên triệu ảnh (ImageNet),
rồi fine-tune cho bài toán của mình.

Cách tiếp cận:
1. **Giai đoạn 1 (Feature Extraction):** Đóng băng EfficientNet, chỉ train lớp đầu ra mới
2. **Giai đoạn 2 (Fine-tuning):** Mở băng một số lớp cuối, train tiếp với LR nhỏ hơn

In [ ]:
# ============================================================
# Xây Transfer Learning Model với EfficientNetB0
# ============================================================
def build_transfer_model(input_shape, num_classes):
    base_model = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    base_model.trainable = False

    inputs = keras.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)

    # [SỬA] Head mạnh hơn: thêm 1 Dense layer + tăng units
    # Lý do: bài toán phân biệt 4 loại u não phức tạp hơn ImageNet
    # → cần head sâu hơn để học được ranh giới phân loại y tế
    x = layers.Dense(512, activation='relu')(x)  # [SỬA] 256→512
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)                   # [SỬA] 0.4→0.5
    x = layers.Dense(256, activation='relu')(x)  # [MỚI] thêm layer 2
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name='EfficientNetB0_Transfer')
    return model, base_model


tl_model, base_model = build_transfer_model((IMG_SIZE[0], IMG_SIZE[1], 3), NUM_CLASSES)

# [SỬA] LR stage 1: 1e-4 → 3e-4
# Lý do: base_model đóng băng hoàn toàn, chỉ train head mới
# → có thể dùng LR cao hơn để hội tụ nhanh hơn
tl_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=3e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f'Tổng tham số       : {tl_model.count_params():,}')
print(f'Tham số có thể train: {sum([np.prod(v.shape) for v in tl_model.trainable_variables]):,}')
print(f'Tham số đóng băng  : {sum([np.prod(v.shape) for v in tl_model.non_trainable_variables]):,}')


In [ ]:
# ============================================================
# GIAI ĐOẠN 1: Feature Extraction — train lớp head mới
# ============================================================
callbacks_tl_stage1 = [
    ModelCheckpoint('best_tl_model.h5', monitor='val_accuracy',
                    save_best_only=True, verbose=1),
    # [SỬA] patience=6 hợp lý: stage 1 chỉ train head, nếu 6 epoch không tăng thì dừng
    EarlyStopping(monitor='val_accuracy', patience=6,
                  restore_best_weights=True, verbose=1),
    # [SỬA] Thêm min_lr=1e-6: tránh LR giảm xuống gần 0 trước khi bước sang Stage 2
    # Nếu không có min_lr, LR cứ nhân 0.5 mãi → đến Stage 2 model gần như đứng yên
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=3, min_lr=1e-6, verbose=1)
]

print('🚀 Giai đoạn 1: Feature Extraction (base model đóng băng)...')
history_tl_s1 = tl_model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator,
    callbacks=callbacks_tl_stage1,
    class_weight=class_weight_dict,
    verbose=1
)
print('\n✅ Giai đoạn 1 hoàn tất!')


In [ ]:
# ============================================================
# STAGE 2: Fine-tuning with explicit progressive unfreeze
# ============================================================
# Important fix: when increasing from last 20 layers to last 80 layers,
# explicitly set the trainable flags. Otherwise layers -80:-20 can remain
# frozen from the previous stage.

base_model.trainable = True

# ---- Stage 2a: unfreeze last 20 layers ----
for layer in base_model.layers:
    layer.trainable = False
for layer in base_model.layers[-20:]:
    if not isinstance(layer, layers.BatchNormalization):
        layer.trainable = True

tl_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-6),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)

callbacks_tl_stage2a = [
    ModelCheckpoint('best_tl_model.h5', monitor='val_accuracy', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-8, verbose=1)
]

print('Stage 2a: unfreeze last 20 non-BN layers, LR=5e-6')
print(f'   Trainable params: {sum([np.prod(v.shape) for v in tl_model.trainable_variables]):,}')
history_tl_s2a = tl_model.fit(
    train_generator,
    epochs=8,
    validation_data=val_generator,
    callbacks=callbacks_tl_stage2a,
    class_weight=class_weight_dict,
    verbose=1
)
print('Stage 2a complete\n')

# ---- Stage 2b: unfreeze last 80 layers ----
for layer in base_model.layers:
    layer.trainable = False
for layer in base_model.layers[-80:]:
    if not isinstance(layer, layers.BatchNormalization):
        layer.trainable = True

# Keep cross-entropy as the default baseline. Combining class_weight and
# focal loss often over-biases small/ambiguous classes on this dataset.
tl_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=['accuracy']
)

callbacks_tl_stage2b = [
    ModelCheckpoint('best_tl_model.h5', monitor='val_accuracy', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-8, verbose=1)
]

print('Stage 2b: unfreeze last 80 non-BN layers, LR=1e-5')
print(f'   Trainable params: {sum([np.prod(v.shape) for v in tl_model.trainable_variables]):,}')
history_tl_s2b = tl_model.fit(
    train_generator,
    epochs=EPOCHS_TL,
    validation_data=val_generator,
    callbacks=callbacks_tl_stage2b,
    class_weight=class_weight_dict,
    verbose=1
)
print('\nStage 2b complete')

In [ ]:
# ============================================================
# Ghép 3 giai đoạn history để vẽ đồ thị liền mạch
# ============================================================
class MergedHistory:
    """Ghép 2 history objects thành 1 để vẽ cùng nhau."""
    def __init__(self, h1, h2):
        self.history = {
            key: h1.history[key] + h2.history[key]
            for key in h1.history
        }

# Ghép cả 3 giai đoạn: Stage1 + Stage2a + Stage2b
history_tl = MergedHistory(MergedHistory(history_tl_s1, history_tl_s2a), history_tl_s2b)
plot_training_history(history_tl, 'EfficientNetB0 Transfer Learning (Progressive Unfreeze)', 'model_tl_history.png')


## 📊 Phần F — So sánh 2 Model trên Validation Set

In [ ]:
# ============================================================
# Quick validation comparison
# Load with compile=False so custom/old loss definitions never break eval.
# Accuracy and loss are computed directly from predictions.
# ============================================================
def predict_metrics(model, generator):
    generator.reset()
    y_prob = model.predict(generator, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    y_true = generator.classes
    y_true_onehot = tf.keras.utils.to_categorical(y_true, num_classes=NUM_CLASSES)
    loss = keras.losses.CategoricalCrossentropy()(y_true_onehot, y_prob).numpy()
    acc = np.mean(y_pred == y_true)
    return float(loss), float(acc)

best_cnn = keras.models.load_model('best_cnn_model.h5', compile=False)
best_tl  = keras.models.load_model('best_tl_model.h5', compile=False)

cnn_val_loss, cnn_val_acc = predict_metrics(best_cnn, val_generator)
tl_val_loss,  tl_val_acc  = predict_metrics(best_tl,  val_generator)

print('=' * 50)
print('VALIDATION COMPARISON')
print('=' * 50)
print(f'  CNN Baseline      : Acc={cnn_val_acc:.4f}  Loss={cnn_val_loss:.4f}')
print(f'  Transfer Learning : Acc={tl_val_acc:.4f}  Loss={tl_val_loss:.4f}')
print()

if tl_val_acc > cnn_val_acc:
    print('Use Transfer Learning for Evaluation')
    BEST_MODEL      = best_tl
    BEST_MODEL_NAME = 'EfficientNetB0'
else:
    print('Use CNN Baseline for Evaluation')
    BEST_MODEL      = best_cnn
    BEST_MODEL_NAME = 'CNN Baseline'

print('=' * 50)

In [ ]:
# ============================================================
# Vẽ so sánh accuracy 2 model
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('So sánh 2 Model', fontsize=14, fontweight='bold')

models_names = ['CNN Baseline', 'Transfer Learning\n(EfficientNetB0)']
val_accs  = [cnn_val_acc,  tl_val_acc]
val_losses = [cnn_val_loss, tl_val_loss]
colors = ['#2196F3', '#4CAF50']

axes[0].bar(models_names, [v*100 for v in val_accs], color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Val Accuracy (%)')
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_ylim([0, 100])
for i, v in enumerate(val_accs):
    axes[0].text(i, v*100 + 1, f'{v*100:.2f}%', ha='center', fontweight='bold')

axes[1].bar(models_names, val_losses, color=colors, edgecolor='white', linewidth=1.5)
axes[1].set_title('Val Loss')
axes[1].set_ylabel('Loss')
for i, v in enumerate(val_losses):
    axes[1].text(i, v + 0.005, f'{v:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n✅ Hoàn tất! → Chuyển sang file 04_Evaluation.ipynb')